# Phase 2.1-2.2 - Schema & Final Models

Two steps that unblock the rest of Phase 2.

### 2.1 - The shot record schema

Decided before any pipeline code, because it constrains everything downstream.

The critical choice: **persist the canonical pose window with every shot**, not just derived scalars. Phase 3 is a coaching layer, and you will think of kinematic features you haven't thought of yet. With the window stored, adding one is a function call; without it, re-extraction costs hours per video.

### 2.2 - Final models

Everything so far is a **LOVO artifact** - 7 detectors and 7 classifiers, each trained on 6 folds, none saved. A tool needs *one* of each, trained on all 12 videos.

> **Important consequence.** The final model has no held-out data by construction. Its performance is *not* measurable - the LOVO scores (detection 0.881, classification 0.792, end-to-end 0.698) are the correct estimate of how it behaves, and those remain the numbers to quote. Any accuracy computed on the training videos after this point is meaningless.

Runs ~1 h on a T4.


## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Setup

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

SEED       = 42
DET_EPOCHS = 40
CLS_EPOCHS = 60
BATCH      = 64
EXCLUDE    = ["test_5"]      # no contact signal in its pose stream

import json, math, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

BASE = Path(BASE); META = BASE/"derived/meta"
STREAM = BASE/"derived/pose_stream"; CKPT = BASE/"models/checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
seed_all(SEED)

def load(stem):
    p = META/f"{stem}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(META/f"{stem}.csv")

strokes = load("strokes")
folds   = json.loads((META/"folds.json").read_text())
PRE, NF = folds["window"]["pre"], folds["window"]["n_frames"]
CLASSES = ["serve", "attack", "control", "defence"]
TECHS   = ["block","chop","flick","lob","loop","push","serve","smash"]
C2I = {c: i for i, c in enumerate(CLASSES)}
T2I = {t: i for i, t in enumerate(TECHS)}

VIDEOS = [v for v in sorted(strokes.video_id.unique(),
          key=lambda x: (x.split("_")[0], int(x.split("_")[1])))
          if v not in EXCLUDE]
GT = {v: {} for v in VIDEOS}
for r in strokes.itertuples():
    if r.video_id in GT:
        GT[r.video_id][int(r.frame_120)] = (r.shot_class, r.side, r.technique)

print(f"{torch.cuda.get_device_name(0)}")
print(f"{len(VIDEOS)} videos, {sum(len(g) for g in GT.values())} strokes")

Tesla T4
11 videos, 1430 strokes


## 3. Shot record schema

Versioned and written to disk so the pipeline and the coaching layer agree on one contract.

`pose_window` is the field that matters - 97 x 17 x 2 canonical keypoints per shot. Everything in `kinematics` is *derived* from it, so new features can be added later without touching video again.

In [3]:
SCHEMA = {
    "version": 1,
    "description": "Per-shot record emitted by analyse(). One row per detected "
                   "stroke. Phase 3 consumes this directly.",
    "identity": {
        "video_id":     "source video stem",
        "rally_id":     "int, 0-based within the video",
        "shot_index":   "int, 0-based within the rally",
        "player":       "'left' | 'right' - which end of the table",
        "frame":        "int, native 120fps frame index of contact",
        "timestamp_s":  "float, seconds from video start",
    },
    "prediction": {
        "shot_class":       "serve | attack | control | defence",
        "class_confidence": "float 0-1, CALIBRATED (see step 2.5)",
        "class_proba":      "list[4], order = CLASSES",
        "technique":        "8-way auxiliary head, lower confidence",
        "abstain":          "bool - true when confidence is below the floor; "
                            "Phase 3 must give no feedback on these",
        "detect_confidence":"float, peak height from the contact detector",
    },
    "pose_window": {
        "shape": [97, 17, 2],
        "note":  "CANONICAL keypoints: hip-centred, torso-scaled, side-mirrored. "
                 "Contact at index 60. THE RAW MATERIAL FOR PHASE 3 - persisted "
                 "so new kinematic features never require re-extraction.",
        "valid": [97, 17],
    },
    "kinematics": {
        "backswing_amplitude": "max wrist distance from hip, pre-contact, torso units",
        "peak_wrist_speed":    "95th pct wrist speed, torso units per frame",
        "time_to_peak":        "frames from contact to peak speed (signed)",
        "contact_height":      "wrist height relative to shoulder line at contact",
        "elbow_angle":         "degrees at contact",
        "elbow_range":         "max minus min elbow angle over the window",
        "trunk_lean":          "degrees from vertical at contact",
        "trunk_rotation":      "shoulder-line angle range over the window",
        "table_distance":      "hip to near table edge at contact, torso units",
        "stance_width":        "ankle separation at contact, torso units",
        "knee_angle":          "mean knee flexion at contact, degrees",
        "follow_through":      "wrist path length post-contact",
        "recovery_time":       "frames until wrist speed returns below 20% of peak",
    },
    "rally": {
        "rally_outcome": "set only on the last shot of a rally, else null",
        "rally_length":  "int, shots in this rally",
    },
    "quality": {
        "pose_confidence": "mean keypoint confidence over the window",
        "detected":        "fraction of window frames with a player box",
    },
}

(META/"shot_record_schema.json").write_text(json.dumps(SCHEMA, indent=2))
print(json.dumps(SCHEMA, indent=2)[:900] + "\n...")
print(f"\n-> {META/'shot_record_schema.json'}")
print(f"\nfields: {sum(len(v) for k, v in SCHEMA.items() if isinstance(v, dict) and k != 'pose_window')}"
      f" + pose_window({NF}x17x2)")

{
  "version": 1,
  "description": "Per-shot record emitted by analyse(). One row per detected stroke. Phase 3 consumes this directly.",
  "identity": {
    "video_id": "source video stem",
    "rally_id": "int, 0-based within the video",
    "shot_index": "int, 0-based within the rally",
    "player": "'left' | 'right' \u2014 which end of the table",
    "frame": "int, native 120fps frame index of contact",
    "timestamp_s": "float, seconds from video start"
  },
  "prediction": {
    "shot_class": "serve | attack | control | defence",
    "class_confidence": "float 0-1, CALIBRATED (see step 2.5)",
    "class_proba": "list[4], order = CLASSES",
    "technique": "8-way auxiliary head, lower confidence",
    "abstain": "bool \u2014 true when confidence is below the floor; Phase 3 must give no feedback on these",
    "detect_confidence": "float, peak height from the contact detector"
  },
...

-> /content/drive/MyDrive/tt_coach/derived/meta/shot_record_schema.json

fields: 29 + pose_win

## 4. Rebuild features

Identical to the Stage 7 construction, which is the version that produced the 0.698 end-to-end result. Channel layout is `kp(34) + vel(34) + valid(17) + table_dist(1) = 86` for the classifier - matching Stage 5 exactly.

In [4]:
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KNE, R_KNE, L_ANK, R_ANK = 11, 12, 13, 14, 15, 16
FLIP = [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]

def canon(kp, sc, seg, mirror):
    kp = kp.astype(np.float32).copy()
    hip = (kp[:, L_HIP] + kp[:, R_HIP]) / 2
    sho = (kp[:, L_SHO] + kp[:, R_SHO]) / 2
    torso = np.linalg.norm(sho - hip, axis=-1)
    scale = np.ones(len(kp), np.float32)
    for s in np.unique(seg):
        m = seg == s; t = torso[m]; t = t[t > 1]
        scale[m] = np.median(t) if len(t) else 1.0
    kp = (kp - hip[:, None, :]) / np.maximum(scale, 1e-3)[:, None, None]
    if mirror:
        kp[..., 0] *= -1; sc = sc.copy()
        for a, b in FLIP:
            kp[:, [a, b]] = kp[:, [b, a]]; sc[:, [a, b]] = sc[:, [b, a]]
    return kp, sc, scale

S = {}
for v in VIDEOS:
    d = np.load(STREAM/f"{v}.npz", allow_pickle=True)
    fidx, seg = d["frame_idx"], d["seg_id"]
    KP, SC, DET = d["keypoints"], d["scores"], d["detected"]
    tb = d["table_box"]
    ch, kps, vals, tds = [], [], [], []
    for pi in (0, 1):
        kp, sc, scale = canon(KP[:, pi], SC[:, pi].astype(np.float32), seg,
                              mirror=(pi == 1))
        vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
        vel[np.diff(seg, prepend=seg[0]) != 0] = 0
        ch += [kp.reshape(len(kp), -1), vel.reshape(len(kp), -1),
               (sc * DET[:, pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc >= 0.35) & DET[:, pi:pi+1])
        hipx = (KP[:, pi, L_HIP, 0] + KP[:, pi, R_HIP, 0]) / 2
        edge = tb[0] if pi == 0 else tb[2]
        tds.append(np.abs(hipx - edge) / np.maximum(scale, 1e-3)
                   if tb[2] > tb[0] else np.zeros(len(kp), np.float32))
    X = np.nan_to_num(np.concatenate(ch, 1).astype(np.float32))

    pos = np.array([i for i in np.where(d["is_contact"])[0]
                    if int(fidx[i]) in GT[v]], int)
    yc = np.zeros(len(fidx), np.float32); t = np.arange(len(fidx))
    ys = np.full(len(fidx), -1.0, np.float32)
    for i in pos:
        lo, hi = max(0, i-8), min(len(fidx), i+9)
        yc[lo:hi] = np.maximum(yc[lo:hi], np.exp(-((t[lo:hi]-i)**2)/8.0))
        ys[max(0, i-2):i+3] = 0.0 if GT[v][int(fidx[i])][1] == "left" else 1.0

    cuts = np.where(np.diff(seg) != 0)[0] + 1
    b = np.concatenate([[0], cuts, [len(seg)]])
    S[v] = dict(X=X, yc=yc, ys=ys, pos=pos, fidx=fidx, seg=seg,
                kp=np.stack(kps, 1), val=np.stack(vals, 1),
                td=np.nan_to_num(np.stack(tds, 1)),
                spans=[(int(b[i]), int(b[i+1])) for i in range(len(b)-1)])
    print(f"  {v}: {len(X):,} frames, {len(pos)} strokes")

C_DET = S[VIDEOS[0]]["X"].shape[1]

def window_at(v, idx, side):
    d = S[v]
    sel = np.clip(np.arange(idx-PRE, idx-PRE+NF), 0, len(d["X"])-1)
    pi = 0 if side == "left" else 1
    kp = d["kp"][sel, pi]; val = d["val"][sel, pi].astype(np.float32)
    vel = np.zeros_like(kp); vel[1:] = np.diff(kp, axis=0)
    td = d["td"][sel, pi][:, None]
    x = np.concatenate([kp.reshape(NF, -1), vel.reshape(NF, -1), val, td], 1)
    return np.nan_to_num(x).T.astype(np.float32)

CX, CY, CT = [], [], []
for v in VIDEOS:
    for i in S[v]["pos"]:
        cls, side, tech = GT[v][int(S[v]["fidx"][i])]
        CX.append(window_at(v, i, side)); CY.append(C2I[cls])
        CT.append(T2I.get(tech, -1))
CX = np.stack(CX); CY = np.array(CY); CT = np.array(CT)
C_CLS = CX.shape[1]
print(f"\ndetector {C_DET} ch | classifier {CX.shape} ({C_CLS} ch)")

  game_1: 20,144 frames, 161 strokes
  game_2: 70,873 frames, 399 strokes
  game_3: 27,911 frames, 153 strokes
  game_4: 24,249 frames, 173 strokes
  game_5: 34,004 frames, 248 strokes
  test_1: 8,027 frames, 84 strokes
  test_2: 2,806 frames, 29 strokes
  test_3: 4,070 frames, 24 strokes
  test_4: 13,251 frames, 71 strokes
  test_6: 5,335 frames, 39 strokes
  test_7: 5,760 frames, 49 strokes

detector 170 ch | classifier (1430, 86, 97) (86 ch)


## 5. Model definitions

In [5]:
class Block(nn.Module):
    def __init__(s, c, d, drop=0.1):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.c2 = nn.Conv1d(c, c, 5, padding=2*d, dilation=d)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.do = nn.Dropout(drop)
    def forward(s, x):
        r = x
        x = s.do(F.gelu(s.n1(s.c1(x))))
        x = s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x + r)

class DetNet(nn.Module):
    def __init__(s, c_in, w=128):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d) for d in (1,2,4,8,16,32,64)])
        s.hc, s.hs = nn.Conv1d(w, 1, 1), nn.Conv1d(w, 1, 1)
    def forward(s, x):
        z = s.blocks(s.stem(x))
        return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s, c):
        super().__init__(); s.score = nn.Conv1d(c, 1, 1)
    def forward(s, x):
        w = torch.softmax(s.score(x), -1)
        return torch.cat([(x*w).sum(-1), x.max(-1).values], -1)

class ClsNet(nn.Module):
    def __init__(s, c_in, w=128):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(c_in, w, 1), nn.BatchNorm1d(w), nn.GELU())
        s.blocks = nn.Sequential(*[Block(w, d, 0.2) for d in (1,2,4,8,16,32)])
        s.pool = AttnPool(w)
        s.trunk = nn.Sequential(nn.Linear(w*2, 256), nn.GELU(), nn.Dropout(0.3))
        s.shot, s.tech = nn.Linear(256, 4), nn.Linear(256, 8)
    def forward(s, x):
        z = s.trunk(s.pool(s.blocks(s.stem(x))))
        return s.shot(z), s.tech(z)

def focal(lg, tg, weight=None, g=2.0):
    m = tg >= 0
    if m.sum() == 0: return lg.sum()*0.0
    lg, tg = lg[m], tg[m]
    ce = F.cross_entropy(lg, tg, weight=weight, reduction="none")
    pt = torch.exp(-F.cross_entropy(lg, tg, reduction="none"))
    return ((1-pt)**g * ce).mean()

print("defined")

defined


## 6. Train the final detector

All 12 videos, no held-out fold. The side loss is included - omitting it in Stage 7 caused ~half the classifier windows to be cut around the wrong player and cost 0.31 macro-F1.

In [6]:
rng = np.random.default_rng(SEED); seed_all(SEED)

cat = np.concatenate([S[v]["X"] for v in VIDEOS])
DMU, DSD = cat.mean(0), cat.std(0) + 1e-6; del cat
eff = np.mean([(S[v]["yc"] > 0.05).mean() for v in VIDEOS])
pw = torch.tensor((1-eff)/eff, device=dev)

det = DetNet(C_DET).to(dev)
opt = torch.optim.AdamW(det.parameters(), lr=2e-3, weight_decay=1e-4)
steps = DET_EPOCHS*40
sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
det.train()
for st in range(steps):
    xs, ys_, ss = [], [], []
    for _ in range(16):
        v = VIDEOS[rng.integers(len(VIDEOS))]; d = S[v]
        a, b = d["spans"][rng.integers(len(d["spans"]))]
        n = b - a
        if n <= 512: sl = slice(a, b); pad = 512-n
        else:
            s0 = a + rng.integers(n-512+1); sl = slice(s0, s0+512); pad = 0
        x = (d["X"][sl]-DMU)/DSD; y = d["yc"][sl]; s_ = d["ys"][sl]
        if pad:
            x = np.pad(x, ((0,pad),(0,0)), mode="edge")
            y = np.pad(y, (0,pad)); s_ = np.pad(s_, (0,pad), constant_values=-1)
        xs.append(x.T); ys_.append(y); ss.append(s_)
    x = torch.tensor(np.stack(xs), dtype=torch.float32, device=dev)
    y = torch.tensor(np.stack(ys_), dtype=torch.float32, device=dev)
    s_ = torch.tensor(np.stack(ss), dtype=torch.float32, device=dev)
    lc, ls = det(x)
    loss = F.binary_cross_entropy_with_logits(lc, y, pos_weight=pw)
    m = (s_ >= 0).float()
    if m.sum() > 0:
        loss = loss + 0.3*(F.binary_cross_entropy_with_logits(
            ls, s_.clamp(min=0), reduction="none")*m).sum()/m.sum()
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(det.parameters(), 1.0); opt.step(); sch.step()
    if st % 400 == 0: print(f"  step {st}/{steps}  loss {loss.item():.4f}")

torch.save({"state": det.state_dict(), "c_in": C_DET,
            "mu": DMU, "sd": DSD, "arch": "DetNet", "seed": SEED,
            "trained_on": VIDEOS, "excluded": EXCLUDE,
            "lovo_f1_at_tol8": 0.881, "lovo_side_acc": 0.994},
           CKPT/"detector_final.pt")
print(f"\n-> {CKPT/'detector_final.pt'}")

  step 0/1600  loss 1.5568
  step 400/1600  loss 0.2006
  step 800/1600  loss 0.1583
  step 1200/1600  loss 0.1570

-> /content/drive/MyDrive/tt_coach/models/checkpoints/detector_final.pt


## 7. Train the final classifier

In [7]:
seed_all(SEED)
xt = torch.tensor(CX, device=dev)
CMU, CSD = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True)+1e-6
xt = (xt-CMU)/CSD
yt = torch.tensor(CY, device=dev); tt = torch.tensor(CT, device=dev)
cnt = np.bincount(CY, minlength=4).clip(1)
cw = torch.tensor(len(CY)/(4*cnt), dtype=torch.float32, device=dev)
p = (1.0/cnt)[CY]; p = p/p.sum()

cls = ClsNet(C_CLS).to(dev)
opt = torch.optim.AdamW(cls.parameters(), lr=2e-3, weight_decay=1e-4)
steps = CLS_EPOCHS*max(1, math.ceil(len(CY)/BATCH))
sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
cls.train()
for st in range(steps):
    b = np.random.choice(len(CY), BATCH, p=p)
    xb = xt[b]
    sh = torch.randint(-6, 7, (len(b),), device=dev)
    ix = (torch.arange(NF, device=dev)[None]+sh[:,None]).clamp(0, NF-1)
    xb = torch.gather(xb, 2, ix[:,None].expand(-1, xb.shape[1], -1))
    xb = xb + torch.randn_like(xb)*0.01
    s_, t_ = cls(xb)
    loss = focal(s_, yt[b], cw) + 0.2*focal(t_, tt[b])
    opt.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(cls.parameters(), 1.0); opt.step(); sch.step()
    if st % 300 == 0: print(f"  step {st}/{steps}  loss {loss.item():.4f}")

torch.save({"state": cls.state_dict(), "c_in": C_CLS,
            "mu": CMU.cpu(), "sd": CSD.cpu(), "arch": "ClsNet",
            "classes": CLASSES, "techniques": TECHS, "seed": SEED,
            "trained_on": VIDEOS, "excluded": EXCLUDE,
            "lovo_macro_f1": 0.792,
            "lovo_per_class": {"serve":0.980,"attack":0.822,
                               "control":0.773,"defence":0.471}},
           CKPT/"classifier_final.pt")
print(f"\n-> {CKPT/'classifier_final.pt'}")

  step 0/1380  loss 1.4396
  step 300/1380  loss 0.1254
  step 600/1380  loss 0.1646
  step 900/1380  loss 0.0067
  step 1200/1380  loss 0.0042

-> /content/drive/MyDrive/tt_coach/models/checkpoints/classifier_final.pt


## 8. Verify the checkpoints load

A load test, not an accuracy test. **Accuracy on these videos is meaningless** - they are the training set. The LOVO numbers stored in each checkpoint are the correct estimate.

In [8]:
for name, Net in [("detector_final.pt", DetNet), ("classifier_final.pt", ClsNet)]:
    ck = torch.load(CKPT/name, map_location=dev, weights_only=False)
    net = Net(ck["c_in"]).to(dev); net.load_state_dict(ck["state"]); net.eval()
    n = sum(p.numel() for p in net.parameters())
    print(f"  {name:<22} {ck['arch']:<8} {ck['c_in']:>3} ch  {n:,} params  OK")

ck = torch.load(CKPT/"classifier_final.pt", map_location=dev, weights_only=False)
net = ClsNet(ck["c_in"]).to(dev); net.load_state_dict(ck["state"]); net.eval()
with torch.no_grad():
    lo, _ = net((torch.tensor(CX[:200], device=dev)-CMU.to(dev))/CSD.to(dev))
    pr = torch.softmax(lo, 1).cpu().numpy()
print(f"\n  smoke test: {pr.shape}, rows sum to "
      f"{pr.sum(1).mean():.3f}, mean max-prob {pr.max(1).mean():.3f}")
print("""
  Reminder: the final models saw every video in training, so any score
  computed here is optimistic and not reportable. Quote the LOVO numbers:
    detection      F1 0.881 @ +/-8 frames, side accuracy 0.994
    classification macro-F1 0.792
    end-to-end     macro-F1 0.698""")

  detector_final.pt      DetNet   170 ch  1,174,658 params  OK
  classifier_final.pt    ClsNet    86 ch  1,068,045 params  OK

  smoke test: (200, 4), rows sum to 1.000, mean max-prob 0.957

  Reminder: the final models saw every video in training, so any score
  computed here is optimistic and not reportable. Quote the LOVO numbers:
    detection      F1 0.881 @ +/-8 frames, side accuracy 0.994
    classification macro-F1 0.792
    end-to-end     macro-F1 0.698


---
## Done

| artifact | purpose |
|---|---|
| `derived/meta/shot_record_schema.json` | the contract between Phase 2 and Phase 3 |
| `models/checkpoints/detector_final.pt` | contact + side, all 12 videos |
| `models/checkpoints/classifier_final.pt` | 4-class + technique, all 12 videos |

Each checkpoint stores its normalisation stats, architecture, training video list, and LOVO scores - so the pipeline never has to guess, and the reported numbers travel with the weights.

Next: **`09_analyse.ipynb`** - the `analyse(video_path)` entry point (steps 2.3-2.6).
